# InternVL3-8B Inference

- 📜 [Read the docs](https://internvl.readthedocs.io/en/latest/internvl2.5/quick_start.html)
- 🔥 [RuntimeError: Tensor.item() cannot be called on meta tensors](https://github.com/OpenGVLab/InternVL/issues/1254)

Rather than adapting a text-only large language model (LLM) into a multimodal large language model (MLLM) that supports visual inputs, InternVL3 jointly acquires multimodal and linguistic capabilities from both diverse multimodal data and pure-text corpora during a single pre-training stage. This unified training paradigm effectively addresses the complexities and alignment challenges commonly encountered in conventional post-hoc training pipelines for MLLMs. To further improve performance and scalability, InternVL3 incorporates variable visual position encoding (V2PE) to support extended multimodal contexts, employs advanced post-training techniques such as supervised fine-tuning (SFT) and mixed preference optimization (MPO), and adopts test-time scaling strategies alongside an optimized training infrastructure. Extensive empirical evaluations demonstrate that InternVL3 delivers superior performance across a wide range of multi-modal tasks. In particular, InternVL3-78B achieves a score of 72.2 on the MMMU benchmark, setting a new state-of-the-art among open-source MLLMs.

![InternVL3-8B Architecture](https://cdn-uploads.huggingface.co/production/uploads/64119264f0f81eb569e0d569/BiiyXN6NOk0p-3rl3ueyL.png)

In [ ]:
import math
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer, AutoConfig
from pathlib import Path
from IPython.display import Image as IPyImage, display
import json

In [ ]:
BASE_DIR = Path.cwd().parent
CATEGORY = "trial"

COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

assert CASE_DIR.is_dir(), "Dataset not found"

In [ ]:
MODEL_PATH = "OpenGVLab/InternVL3-1B"

TORCH_DTYPE = torch.bfloat16
LOAD_IN_4BIT = True
LOAD_IN_8BIT = False

LOW_CPU_MEM_USAGE = True
USE_FLASH_ATTN = True
TRUST_REMOTE_CODE = True

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

In [ ]:
def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose(
        [
            T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
            T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(mean=MEAN, std=STD),
        ]
    )
    return transform


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio


def dynamic_preprocess(
    image, min_num=1, max_num=12, image_size=448, use_thumbnail=False
):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j)
        for n in range(min_num, max_num + 1)
        for i in range(1, n + 1)
        for j in range(1, n + 1)
        if i * j <= max_num and i * j >= min_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size,
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images


def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert("RGB")
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(
        image, image_size=input_size, use_thumbnail=True, max_num=max_num
    )
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

In [ ]:
def split_model(model_name):
    device_map = {}
    world_size = torch.cuda.device_count()
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    num_layers = config.llm_config.num_hidden_layers
    # Since the first GPU will be used for ViT, treat it as half a GPU.
    num_layers_per_gpu = math.ceil(num_layers / (world_size - 0.5))
    num_layers_per_gpu = [num_layers_per_gpu] * world_size
    num_layers_per_gpu[0] = math.ceil(num_layers_per_gpu[0] * 0.5)
    layer_cnt = 0
    for i, num_layer in enumerate(num_layers_per_gpu):
        for j in range(num_layer):
            device_map[f"language_model.model.layers.{layer_cnt}"] = i
            layer_cnt += 1
    device_map["vision_model"] = 0
    device_map["mlp1"] = 0
    device_map["language_model.model.tok_embeddings"] = 0
    device_map["language_model.model.embed_tokens"] = 0
    device_map["language_model.output"] = 0
    device_map["language_model.model.norm"] = 0
    device_map["language_model.model.rotary_emb"] = 0
    device_map["language_model.lm_head"] = 0
    device_map[f"language_model.model.layers.{num_layers - 1}"] = 0

    return device_map

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH, trust_remote_code=True, use_fast=False
)

model = AutoModel.from_pretrained(
    MODEL_PATH,
    torch_dtype=TORCH_DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=LOAD_IN_8BIT,
    low_cpu_mem_usage=LOW_CPU_MEM_USAGE,
    use_flash_attn=USE_FLASH_ATTN,
    trust_remote_code=TRUST_REMOTE_CODE,
    # device_map=split_model(MODEL_PATH),
).eval()

In [ ]:
# set the max number of tiles in `max_num`
image_path = (
    CASE_DIR
    / "atomic-layer-etching"
    / "experimental-usecase"
    / "16"
    / "images"
    / "fig1.jpg"
)
display(IPyImage(image_path))

In [ ]:
pixel_values = load_image(image_path, max_num=12).to(torch.bfloat16).cuda()
generation_config = dict(max_new_tokens=1024, do_sample=True)

## 1. Single Figure Classification

A supervised multi-class image classification task. Systems must identify one of the 49 figure classes. 

- Inputs
    - A scientific figure extracted from an ALD/E research paper.
    - Optional metadata, such as figure caption text, may be provided as context.
- Output
    - Predicted class labels corresponding to one of the 49 figure types.

In [ ]:
prompt1 = """
<image>

Classify each detected subfigure (a, b, c, etc.) into exactly one category from the 49-class taxonomy.

Strict requirements:

1. Detect each distinct plot (subfigure a, b, c, etc.).
2. Process only valid chart regions (axes, ticks, labels, legends, plotted data).
3. Each detected plot must be classified with exactly one label.
4. Use only the predefined class labels below:
  * **area chart**: Filled area under a line to show cumulative values or trends
  * **bar chart**: Rectangular bars to compare quantities across categories
  * **3d bar chart**: Bar chart displayed in three dimensions
  * **grouped bar chart**: Bars grouped by categories for side-by-side comparison
  * **stacked bar chart**: Bars stacked to show part-to-whole relationships
  * **box plot**: Statistical distribution showing median, quartiles, and outliers
  * **bubble chart**: Scatter plot with variable marker size representing a third dimension
  * **donut chart**: Pie chart with a central hole to show proportions
  * **funnel chart**: Progressive reduction across stages of a process
  * **heatmap**: Matrix of values represented with colors
  * **line chart**: Continuous line showing trends over intervals
  * **multiple line chart**: Several lines showing multiple series of trends
  * **multi-axis chart**: Plot with multiple axes to compare different scales
  * **pie chart**: Circular chart divided into slices to show proportions
  * **polar chart (rose chart)**: Circular chart plotting values by angle
  * **radar chart (spider chart)**: Multivariate data represented in a radial layout
  * **3d scatter plot**: Scatter plot displayed in three dimensions
  * **scatter plot**: Points plotted on two axes to show correlations
  * **multiple scatter plot**: Points plotted on two axes to show correlations allows and to visualize different data sets/series/groups on the same chart.
  * **treemap**: Nested rectangles sized by values to show hierarchy
  * **spectra chart**: Specialized line chart used in scientific spectroscopy/diffraction contexts (NMR, IR, Raman, UV-vis, MS, XRD)
  * **stacked spectra chart**: Specialized stacked multiple-line chart used in scientific spectroscopy/diffraction contexts (NMR, IR, Raman, UV-vis, MS, XRD). Used to visualize multiple spectra in a single plot, allowing for easy comparison of peak shifts, changes in peak splittings, and signal intensities.
  * **multi spectra chart**: Specialized multiple-line chart used in scientific spectroscopy/diffraction contexts (NMR, IR, Raman, UV-vis, MS, XRD). Used to visualize multiple spectra in a single plot, allowing for easy comparison of peak shifts, changes in peak splittings, and signal intensities.
  * **phase diagram**: Specialized chart showing equilibrium phase boundaries in temperature–pressure–composition space
  * **band diagram**: Specialized chart plotting electronic energy levels vs. momentum (k) or position, showing band gaps and Fermi levels
  * **adsorption isotherm**: Specialized line/scatter plot showing gas uptake vs. pressure (or relative pressure), used to derive Henry constants and capacity values
  * **process timing diagram**: Time-axis plot showing one or more process variables (e.g., gas flows, pressure, power, valve states) as step-like or pulsed functions over a cycle or sequence of steps.
  * **contour heatmap**: Profile mapping of pressure/temperature or any relevant parameter of study
  * **image panel**: Collection of microscopy or spectroscopy images
  * **map/geo chart**: Geographic or spatial distribution visualization
  * **competing reaction rate curve**: Uses abstract curves, labels, and shaded regions to visually explain the scientific concept alone in this ALE/ALD process.
  * **molecular structure diagram**: Chemical structure drawings of molecules or precursors
  * **reaction scheme**: Arrows and molecules showing chemical reactions
  * **reaction energy profile diagram**: Pathways showing relative energies of reactant complexes
  * **process flow diagram**: Schematic showing sequential or cyclic steps in a scientific or technical process
  * **apparatus diagram**: Diagram of experimental or laboratory setups
  * **conceptual diagram**: Illustration of theoretical models or mechanisms
  * **device structure**: Illustration of theoretical models or mechanisms
  * **chromaticity diagram**: A chromaticity diagram represents the chromaticity of colors, which is defined by two parameters: hue and saturation (or colorfulness). It allows for the visualization of color relationships and is essential in color science for understanding how colors interact and can be reproduced.
  * **periodic table map**: Property overlay aligned to the full periodic table layout (rows, groups, blocks), typically showing trends across all or most elements
  * **element–property matrix**: Matrix-style visualization linking a subset of elements (e.g. lanthanides) with categorical or binary properties (e.g. precursor availability)
  * **network diagram**: Nodes and edges showing relationships or interactions
  * **tree diagram**: Hierarchical branching structure (taxonomy, phylogeny, decision)
  * **workflow diagram**: Diagram showing pipeline or methodological steps
  * **timeline chart**: Chronological sequence of events or steps
  * **comparison table**: Structured tabular comparison of properties or studies
  * **formula**: Mathematical or chemical expression typeset as formula
  * **table**: General tabular data representation
  * **unknown**: Unclassified or unclear figure type
5. Base classification solely on visual evidence.
6. If multiple classes seem plausible, choose the most visually dominant category.
7. Do not invent new labels.
8. Do NOT output duplicated keys.
9. Do NOT output raw JSON fragments, debugging text, or multiple formatting styles.
10. Output valid JSON only, with no code fences or surrounding text.

Output Schema:

{
  "type": "object",
  "additionalProperties": {
    "type": "string",
    "enum": [
      "Spectra Chart",
      "Stacked Spectra Chart",
      "Multi Spectra Chart",
      "Phase Diagram",
      "Band Diagram",
      "Adsorption Isotherm",
      "Process timing diagram",
      "Contour Heatmap",
      "Area Chart",
      "Bar Chart",
      "3D Bar Chart",
      "Grouped Bar Chart",
      "Stacked Bar Chart",
      "Box Plot",
      "Bubble Chart",
      "Donut Chart",
      "Funnel Chart",
      "Heatmap",
      "Line Chart",
      "Multiple Line Chart",
      "Multi-axis Chart",
      "Pie Chart",
      "Polar Chart (Rose Chart)",
      "Radar Chart (Spider Chart)",
      "3D Scatter Plot",
      "Scatter Plot",
      "Multiple Scatter Plot",
      "Treemap",
      "Molecular Structure Diagram",
      "Reaction Scheme",
      "Process Flow Diagram",
      "Apparatus Diagram",
      "Conceptual Diagram",
      "Competing Reaction Rate Curve",
      "Device Structure",
      "Chromaticity Diagram",
      "Reaction Energy Profile Diagram",
      "Periodic Table Map",
      "Element–property Matrix",
      "Network Diagram",
      "Tree Diagram",
      "Image Panel",
      "Map/Geo Chart",
      "Workflow Diagram",
      "Timeline Chart",
      "Comparison Table",
      "Formula",
      "Table"
    ]
  },
  "minProperties": 1
}

Example Output:

{"a":"Line Chart","b":"Multiple Line Chart"}

No explanations.
No markdown.
No code fences.
No trailing commas.
Valid JSON only.
"""

response = model.chat(tokenizer, pixel_values, prompt1, generation_config)

In [ ]:
json.loads(response)

## 2. Single Figure Data Table Extraction

This task focuses on structured reconstruction of the underlying tabular data encoded in scientific charts. Systems must identify column/field labels, table structure, and the textual or numerical values in each cell, producing a machine-readable Markdown representation of the data shown in the quantitative chart.

- Input
    - A scientific chart or plot image (e.g., bar chart, line chart, scatter plot, spectra chart, phase diagram).
- Output
    - A Markdown-formatted table containing:
    - Field/column names of the visualized entities
    - Extracted textual and/or numeric cell values

In [ ]:
prompt2 = """
<image>

Extract structured tabular data from all quantitative plots in the image.

Strict requirements:

1. Detect each distinct plot (subfigure a, b, c, etc.).
2. Process only valid chart regions (axes, ticks, labels, legends, plotted data).
3. Ignore decorative graphics, schematics, arrows, background elements, and repeated fragments.
4. Do NOT extract isolated text snippets or axis tick labels as standalone plots.
5. Each detected plot must produce exactly ONE complete Markdown table.
6. Tables must contain:
   - Column headers derived from axis labels and legend entries.
   - Reconstructed numeric or textual values inferred from plotted data.
7. Do NOT output partial tables.
8. Do NOT output duplicated keys.
9. Do NOT output raw JSON fragments, debugging text, or multiple formatting styles.
10. Output valid JSON only, with no code fences or surrounding text.

Output schema:

{
  "type": "object",
  "additionalProperties": {
    "type": "string"
  },
  "minProperties": 1,
  "description": "Mapping from plot identifiers (e.g., 'a', 'b', 'c') to Markdown tables. Each value must be a single complete Markdown table string."
}

Example output:

{"a": "| Col1 | Col2 |\n|---|---|\n| v1 | v2 |", "b": "| Col1 | Col2 |\n|---|---|\n| v1 | v2 |"}

No explanations.
No markdown.
No code fences.
No trailing commas.
Valid JSON only.
"""

response = model.chat(tokenizer, pixel_values, prompt2, generation_config)

In [ ]:
json.loads(response)

## 3. Single Figure Summarization

The goal of this task is to generate concise, factual summaries that capture the key trends, relationships, and scientific insights presented in the figure. Systems must demonstrate accurate semantic interpretation grounded in the visual content, possibly supported by the caption when available.

- Input
    - A scientific chart or plot image.
    - figure caption text (optional).
- Output
    - A short textual summary (1–3 sentences) describing the main trends and takeaways.

In [ ]:
prompt3 = """
<image>

Generate a concise, factual, brief (1-3 sentences) summary that captures the key trends, relationships, and scientific insights presented in the figure.

Strict requirements:

1. Detect each distinct plot (subfigure a, b, c, etc.).
2. Process only valid chart regions (axes, ticks, labels, legends, plotted data).
3. Ignore decorative graphics, schematics, arrows, background elements, and repeated fragments.
4. Base the summary strictly on visible chart data (and caption if provided).
5. Do not speculate beyond the visual evidence.
6. Each detected plot must produce exactly ONE summary.
7. No decorative or stylistic commentary.
8. Do NOT output duplicated keys.
9. Do NOT output raw JSON fragments, debugging text, or multiple formatting styles.
10. Output valid JSON only, with no code fences or surrounding text.

Output Schema:

{
  "type": "object",
  "additionalProperties": {
    "type": "string",
    "minLength": 1
  },
  "minProperties": 1,
  "description": "Mapping from plot identifiers (e.g., 'a', 'b', 'c') to concise data-driven summaries (1–3 sentences each)."
}

Example Output:

{"a":"summary for plot a","b":"summary for plot b"}

No explanations.
No markdown.
No code fences.
No trailing commas.
Valid JSON only.
"""

response = model.chat(tokenizer, pixel_values, prompt3, generation_config)

In [ ]:
json.loads(response)

## 4. Single Figure VQA

This task tests fine-grained reasoning over scientific figures by requiring systems to answer natural-language questions that reference the visual content, including axes, legends, and data patterns. The VQA task is divided into four scientifically meaningful sub-tasks:

- **Process-Oriented**: Assesses understanding of ALD/E cycles, precursor chemistry, and reaction mechanisms.
- **Comparative/Trend**: Evaluates reasoning about how experimental variables (temperature, cycle count, pulse length, etc.) influence outcomes such as growth rate or film thickness.
- **Structure-Property**: Tests the ability to link precursor families or chemical structures to material properties such as thermal stability or growth characteristics etc.
- **Application/Performance**: Measures reasoning related to device-relevant outcomes such as luminescence behavior or photovoltaic performance etc.

- Inputs
    - A scientific chart or plot image
    - A natural-language question (e.g., "At what temperature does the maximum growth rate occur?")
- Outputs
    - Yes/No: "yes" or "no"
    - Factoid: a textual term (e.g., "O₂ plasma")
    - List: comma-separated values (order-insensitive)
    - Paragraph: ≥ 3 sentences providing an explanatory answer

In [ ]:
prompt_template = """
<image>

Generate answers to the provided scientific visual question(s) by reasoning strictly over the information visible in the figure.

Strict requirements:

1. Detect each distinct plot (subfigure a, b, c, etc.).
2. Process only valid chart regions (axes, ticks, labels, legends, plotted data).
3. Ignore decorative graphics, schematics, arrows, background elements, and repeated fragments.
4. Answer each question based only on visible chart data and annotations.
5. Do not speculate beyond the visual evidence.
6. Use the specified answer type:
   - “Yes/No”: respond with either “yes” or “no”.
   - “Factoid”: concise term or short phrase.
   - “List”: comma-separated values (order-insensitive).
   - “Paragraph”: at least three sentences of explanation.
7. For comparative or trend questions, reference only observable relationships.
8. No stylistic commentary or extraneous explanations.
9. Do NOT output duplicated keys.
10. Do NOT output raw JSON fragments, debugging text, or multiple formatting styles.
11. Output valid JSON only, with no code fences or surrounding text.

Output Schema:

{{
  "type": "object",
  "additionalProperties": {{
    "type": "array",
    "items": {{
      "type": "object",
      "required": ["question_type", "questions", "answer_type", "answer"],
      "properties": {{
        "question_type": {{ "type": "string" }},
        "questions": {{ "type": "string" }},
        "answer_type": {{ "type": "string", "enum": ["Yes/No", "Factoid", "List", "Paragraph"] }},
        "answer": {{ "type": "string", "minLength": 1 }}
      }},
      "additionalProperties": false
    }}
  }},
  "minProperties": 1,
  "description": "Mapping from plot identifiers (e.g., 'a', 'b', 'c') to arrays of question-answer objects."
}}

Example Output:

{{"a":[{{"question_type":"Yes/No","questions":"Is trend increasing?","answer_type":"Yes/No","answer":"yes"}}]}}

No markdown.
No code fences.
No trailing commas.
Valid JSON only.

Question: {question}
"""

In [ ]:
response, history = model.chat(
    tokenizer,
    pixel_values,
    prompt_template.format(question="Why are there negative values?"),
    generation_config,
    history=None,
    return_history=True,
)

In [ ]:
json.loads(response)

In [ ]:
response, history = model.chat(
    tokenizer,
    pixel_values,
    prompt_template.format(
        question="What material has the lowest and highest deposition rates during the process?"
    ),
    generation_config,
    history=history,
    return_history=True,
)

In [ ]:
json.loads(response)

## Batch Inference

In [ ]:
image_path_1 = (
    CASE_DIR
    / "atomic-layer-etching"
    / "experimental-usecase"
    / "16"
    / "images"
    / "fig1.jpg"
)
image_path_2 = (
    CASE_DIR
    / "atomic-layer-etching"
    / "experimental-usecase"
    / "16"
    / "images"
    / "fig_2.jpg"
)

pixel_values1 = load_image(image_path_1, max_num=12).to(torch.bfloat16).cuda()
pixel_values2 = load_image(image_path_2, max_num=12).to(torch.bfloat16).cuda()
num_patches_list = [pixel_values1.size(0), pixel_values2.size(0)]
pixel_values = torch.cat((pixel_values1, pixel_values2), dim=0)

In [ ]:
prompts = [prompt2] * len(num_patches_list)
responses = model.batch_chat(
    tokenizer,
    pixel_values,
    num_patches_list=num_patches_list,
    questions=prompts,
    generation_config=generation_config,
)
for response in responses:
    print(json.loads(response))